Efficient attention, multilayer, custom positional encoder and NanoBert encoder size reduction. SwiGLU layer instead of simple FFN

In [ ]:
%%capture
!pip install datasets

### Imports

In [ ]:
import torch
import torch.nn as nn

from keras.preprocessing.sequence import pad_sequences
import numpy as np

import torch.optim as optim
import time
import random
import math

from tqdm import tqdm


### Embedding Layer (Word + Positional)

Instead of using a sinusoidal positional encoding (as in the original paper), a learnable positional embedding is used.

In [ ]:
def base_3_conversion(number):
    # Funzione per la conversione di un numero in base 10 in base 3
    quotient, remainder = divmod(number,3)
    result = [remainder]
    while quotient > 0:
        quotient, remainder = divmod(quotient,3)
        result.append(remainder)  # Inserisci il resto all'inizio della lista
    return result

def base_3_list(n):
    # Funzione per ottenere una lista di liste dei numeri da 0 a 9 convertiti in base 3
    result = []
    for num in range(n):
        converted_num = base_3_conversion(num)
        result.append(converted_num)
    return result


In [ ]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, max_length, embed_dim, reduced_embed = 16, dropout=0.1):
    super(Embedding, self).__init__()

    log_len = math.ceil(math.log(max_length) / math.log(3))
    self.word_embed = nn.Embedding(vocab_size, reduced_embed)
    self.expand_layer = nn.Linear(reduced_embed, embed_dim - log_len)
    # self.word_embed = nn.Embedding(vocab_size, embed_dim - log_len)

    base_3_representation = base_3_list(max_length)
    self.pos_embed = torch.tensor(pad_sequences(base_3_representation, maxlen=log_len, truncating="post", padding="post", dtype="int") - 1)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    batch_size, seq_length = x.shape
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    word_embeddings = self.expand_layer(self.word_embed(x))
    pos_embeddings = self.pos_embed.unsqueeze(0).repeat(x.size(0), 1, 1).to(device)
    embedding = torch.cat((word_embeddings, pos_embeddings), dim = 2)
    return self.dropout(embedding)

### Multi-Head Self-Attention

<center>
<img src="https://raw.githubusercontent.com/HosseinZaredar/Transformer-from-Scratch/main/SelfAttention.png" width="600" align="center"/>
</center>


In [ ]:
class EfficientAttention(nn.Module):
  def __init__(self, embed_dim, num_heads):
    super(EfficientAttention, self).__init__()
    self.embed_dim = embed_dim
    # self.num_heads = num_heads
    # self.head_dim = embed_dim // num_heads

    # assert (self.num_heads*self.head_dim == self.embed_dim),'embed size must be divisible by number of heads'

    self.w_queries = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
    self.w_output = nn.Linear(self.embed_dim, self.embed_dim, bias=False)


  def forward(self, x):

    # shape of x = [batch_size, sentence_length, embedding_dim]
    batch_size = x.shape[0]
    sentence_len = x.shape[1]

    queries = self.w_queries(x) #shape [batch_size, sentence_length, embedding_dim]

    attention_scores = torch.einsum('bij,bjk->bik', queries, torch.transpose(x,1,2))

    attention_dist = torch.softmax(attention_scores / (self.embed_dim ** (1/2)), dim=-1)

    attention_out = torch.einsum('bij,bjk->bik', attention_dist, x)

    out = self.w_output(attention_out)

    return out

### Transformer Encoder

<center>
<img src="https://raw.githubusercontent.com/HosseinZaredar/Transformer-from-Scratch/main/Encoder.png" width="200" align="center"/>
</center>

In [ ]:
class TransformerEncoder(nn.Module):
  def __init__(self, embed_dim, num_heads, forward_expansion, dropout=0.1):
    super(TransformerEncoder, self).__init__()

    self.attention = EfficientAttention(embed_dim, num_heads)
    self.norm1 = nn.LayerNorm(embed_dim)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.fc_up1 = nn.Linear(embed_dim, int(forward_expansion*embed_dim))
    self.fc_up2 = nn.Linear(embed_dim, int(forward_expansion*embed_dim))
    self.fc_down = nn.Linear(int(forward_expansion*embed_dim), embed_dim)
    self.silu = torch.nn.SiLU(inplace=False)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    attention_out = self.dropout(self.attention(x))
    x = self.norm1(x + attention_out)
    mid1 = self.fc_up1(x)
    mid2 = self.silu(self.fc_up2(x))
    mid = torch.mul(mid1,mid2)
    swiglu = self.fc_down(mid)
    forward_out = self.dropout(swiglu)
    out = self.norm2(x + forward_out)

    return out

### End-to-End Classifier

1. An embedding layer
2. A single transformer encoder layer
3. A fully-connected network as a linear classifier

In [ ]:
class Classifier(nn.Module):
  def __init__(self, vocab_size, max_length, red_embed_dim, embed_dim, num_heads, forward_expansion, layers):
      super(Classifier, self).__init__()

      self.embedder = Embedding(vocab_size, max_length, embed_dim, red_embed_dim)
      blocks = []
      blocks += [TransformerEncoder(embed_dim, num_heads, forward_expansion) for _ in range(layers)]
      self.encoder = nn.Sequential(*blocks)
      self.fc = nn.Linear(embed_dim, 1)

  def forward(self, x):
    embedding = self.embedder(x)
    encoding = self.encoder(embedding)
    compact_encoding = encoding.max(dim=1)[0]
    out = self.fc(compact_encoding)
    return torch.sigmoid(out)

In [ ]:
# Print the model size
def print_model_size(model):
  param_size = 0
  param_count = 0
  for param in model.parameters():
    param_size += param.nelement() * param.element_size()
    param_count += param.nelement()
  buffer_size = 0
  for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

  size_all_mb = (param_size + buffer_size) / 1024**2
  print('Model params: {:.3f}M'.format(param_count/1e6))
  print('Model size: {:.3f}MB'.format(size_all_mb))

### Load and Preprocess IMDb Dataset

In [ ]:
VOCAB_SIZE = 512*4

In [ ]:
from datasets import load_dataset
import sentencepiece as spm
import os

#load dataset
dataset = load_dataset("imdb")
train_data = dataset['train']
test_data = dataset['test']


text_train = train_data.to_dict()["text"]
label_train = train_data.to_dict()["label"]

text_test = test_data.to_dict()["text"]
label_test = test_data.to_dict()["label"]

assert len(text_train) == len(label_train)
assert len(text_test) == len(label_test)

if not os.path.exists("./train_ds.txt"):
  #reduce dataset for sentencepiece training
  idxs = random.sample(range(len(text_train)), 10000)
  aux = [text_train[i] for i in idxs]
  #save file of dataset for tokenizer
  filename = "./train_ds.txt"
  with open(filename, 'w') as f:
    for s in aux:
      f.write(s)

if not os.path.exists("./m.model"):
  #Train tokenizer
  spm.SentencePieceTrainer.train(input='train_ds.txt', model_prefix='m', max_sentence_length = 100000000 ,vocab_size=VOCAB_SIZE)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
#Load tokenizer
sp = spm.SentencePieceProcessor(model_file='m.model')

print(f'Dictionary size {sp.get_piece_size()}')
print(f'Vocabulary: {[sp.id_to_piece(id) for id in range(sp.get_piece_size())]}')
print(f'Encoding results:  {sp.encode("this is a phrase that could be commonly found", out_type=str)} -> {sp.encode("this is a phrase that could be commonly found")}')

Dictionary size 2048
Vocabulary: ['<unk>', '<s>', '</s>', 's', '▁the', '.', ',', '▁a', '▁and', '▁of', '▁to', 't', '▁', "'", '▁is', 'br', '▁in', '▁/>', '<', 'ing', '▁it', 'ed', '▁I', '▁that', 'y', 'd', '▁this', 'n', 'ly', '-', '▁was', '▁for', 'e', '▁movie', '▁with', '▁film', 'er', '▁as', 're', '▁The', '▁but', '▁on', 'o', '▁be', 'a', '▁(', 'm', 'c', '▁you', '▁"', '"', '▁are', 'al', 'r', '▁he', '▁not', 'p', 'i', '▁have', '▁his', 'g', 'or', 'in', '▁one', '▁an', ')', '▁who', '▁so', '▁all', '▁by', 'l', 'le', 'I', 'ri', 'ar', '▁from', '▁like', '▁at', 'ur', '▁out', 'en', '▁her', 'u', '▁about', 'an', '▁they', 'it', 'b', 'f', '▁or', 'k', '▁just', '▁re', '▁has', '▁f', '▁S', '!', '▁A', 'h', '▁me', 'ic', 'w', 'es', 'th', 'A', '▁b', '▁It', '▁p', 've', '▁can', '▁B', 'on', '▁some', 'v', '▁no', '?', '▁good', '▁c', '▁up', '▁more', 'The', 'S', '▁would', '▁very', '▁what', '▁F', 'at', '▁time', '▁do', '▁C', '...', 'ra', '▁there', 'll', '▁she', '▁when', '▁see', 'il', '▁story', 'ch', 'se', '▁even', '▁T', '▁we

In [ ]:
# #reduce amount of data
# idxs = random.sample(range(len(text_train)), 2000)
# text_train = [text_train[i] for i in idxs]
# label_train = [label_train[i] for i in idxs]

# idxs = random.sample(range(len(text_test)), 500)
# text_test = [text_test[i] for i in idxs]
# label_test = [label_test[i] for i in idxs]

# assert len(text_train) == len(label_train)
# assert len(text_test) == len(label_test)

In [ ]:
train_tokens = list(map(lambda t: [1] + sp.encode(t)[:510] + [2], text_train))
test_tokens = list(map(lambda t: [1] + sp.encode(t)[:510] + [2], text_test))

In [ ]:
train_tokens_ids = pad_sequences(train_tokens, maxlen=512, truncating="post", padding="post", dtype="int")
test_tokens_ids = pad_sequences(test_tokens, maxlen=512, truncating="post", padding="post", dtype="int")

In [ ]:
train_masks = [[float(i > 0) for i in ii] for ii in train_tokens_ids]
test_masks = [[float(i > 0) for i in ii] for ii in test_tokens_ids]

In [ ]:
train_tokens_tensor = torch.tensor(train_tokens_ids)
train_y_tensor = torch.tensor(np.array(label_train).reshape(-1, 1)).float()

test_tokens_tensor = torch.tensor(test_tokens_ids)
test_y_tensor = torch.tensor(np.array(label_test).reshape(-1, 1)).float()

train_masks_tensor = torch.tensor(train_masks)
test_masks_tensor = torch.tensor(test_masks)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

train_dataset = TensorDataset(train_tokens_tensor, train_masks_tensor, train_y_tensor)
train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=16)

test_dataset = TensorDataset(test_tokens_tensor, test_masks_tensor, test_y_tensor)
test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(test_dataset, sampler=test_sampler, batch_size=16)

### Initializing The Model

In [ ]:
REDUCED_EMBEDDING_DIM = 16
EMBED_DIM = 124
NUM_HEADS = 8
FORWARD_EXPANSION = 0.1
MAX_LENGTH = 512
LAYERS = 4

#initialize model
classifier = Classifier(VOCAB_SIZE, MAX_LENGTH, REDUCED_EMBEDDING_DIM, EMBED_DIM, NUM_HEADS, FORWARD_EXPANSION, LAYERS)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classifier.to(device)

#print all model parameters with names
for name, param in classifier.named_parameters():
  print(f"{name}: {param.nelement()}")

#print the model size
print_model_size(classifier)

#63488 dic 512 dim 124

embedder.word_embed.weight: 32768
embedder.expand_layer.weight: 1888
embedder.expand_layer.bias: 118
encoder.0.attention.w_queries.weight: 15376
encoder.0.attention.w_output.weight: 15376
encoder.0.norm1.weight: 124
encoder.0.norm1.bias: 124
encoder.0.norm2.weight: 124
encoder.0.norm2.bias: 124
encoder.0.fc_up1.weight: 1488
encoder.0.fc_up1.bias: 12
encoder.0.fc_up2.weight: 1488
encoder.0.fc_up2.bias: 12
encoder.0.fc_down.weight: 1488
encoder.0.fc_down.bias: 124
encoder.1.attention.w_queries.weight: 15376
encoder.1.attention.w_output.weight: 15376
encoder.1.norm1.weight: 124
encoder.1.norm1.bias: 124
encoder.1.norm2.weight: 124
encoder.1.norm2.bias: 124
encoder.1.fc_up1.weight: 1488
encoder.1.fc_up1.bias: 12
encoder.1.fc_up2.weight: 1488
encoder.1.fc_up2.bias: 12
encoder.1.fc_down.weight: 1488
encoder.1.fc_down.bias: 124
encoder.2.attention.w_queries.weight: 15376
encoder.2.attention.w_output.weight: 15376
encoder.2.norm1.weight: 124
encoder.2.norm1.bias: 124
encoder.2.norm2.weight: 12

### Training

In [ ]:
optimizer = optim.Adam(classifier.parameters(), lr=2e-4)

In [ ]:
criterion = nn.BCELoss()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion.to(device);

In [ ]:
from tqdm import tqdm


for epoch in range(5):
    classifier.train()
    train_loss = 0
    tqdm_train_loader = tqdm(train_dataloader, desc=f"Epoch {epoch+1}", leave=False)

    for step_num, batch_data in enumerate(tqdm_train_loader):

        token_ids, masks, labels = tuple(t.to(device) for t in batch_data)

        logits = classifier(token_ids)
        # print(logits)
        # print(labels)

        batch_loss = criterion(logits, labels)
        train_loss += batch_loss.item()

        classifier.zero_grad()
        batch_loss.backward()


        nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
        optimizer.step()

        log_step = 50
        if step_num % log_step == (log_step - 1):
          tqdm_train_loader.set_postfix(loss = train_loss / log_step)
          train_loss = 0


### Evaluation

In [ ]:
classifier.eval()
bert_predicted = []
all_logits = []

tqdm_test_loader = tqdm(test_dataloader, desc=f"Evaluation: ", leave=False)

with torch.no_grad():
    for step_num, batch_data in enumerate(tqdm_test_loader):

        token_ids, masks, labels = tuple(t.to(device) for t in batch_data)

        logits = classifier(token_ids)
        loss = criterion(logits, labels)
        numpy_logits = logits.cpu().detach().numpy()

        bert_predicted += list(numpy_logits[:, 0] > 0.5)
        all_logits += list(numpy_logits[:, 0])

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(label_test, bert_predicted))
#dic = 512; Embed_sz = 128; Layers = 4 --> 75% f1 score
#dic = 512 * 8; Embed_sz = 128; Layers = 4 --> 82% f1 score
#dic = 512 * 8; Embed_sz = 128; Layers = 1 --> 83% f1 score
#dic = 512 * 8; Embed_sz = 128; Layers = 10 --> 83% f1 score Doppio training cycle
#dic = 512 * 4; Embed_sz = 128; Layers = 10 --> 83% f1 score Doppio training cycle

              precision    recall  f1-score   support

           0       0.81      0.85      0.83     12500
           1       0.84      0.79      0.82     12500

    accuracy                           0.82     25000
   macro avg       0.82      0.82      0.82     25000
weighted avg       0.82      0.82      0.82     25000



In [ ]:
from sklearn.metrics import matthews_corrcoef
matthews_corrcoef(label_test, bert_predicted)

0.6566755139721194

In [ ]:
# custom_text = "I really liked this movie, it was for sure worth watching. The color correction was sublime, It really gave a lot to the film"
# custom_text = "This film sucked, I hated it, it was a waste of time. All actors were terrible and even the lights were terrible. Not going for a rewatch"
enc = [1] + sp.encode(custom_text)[:510] + [2]
enc = enc + [0]* (512-len(enc))
enc_tensor = torch.tensor(enc).unsqueeze(0).to(device)
classifier(enc_tensor)

NameError: name 'custom_text' is not defined